# Week 4 — Baseline Linear Regression Model

In this notebook, I trained a baseline Linear Regression model to predict California residential single-family property close prices. The goal of this baseline model is to create an initial performance benchmark before testing more advanced models in later weeks.


In [1]:
import pandas as pd
import numpy as np

from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

In [6]:
train_no_outliers = pd.read_csv("train_residential_single_family_week3_no_outliers.csv")
test_no_outliers = pd.read_csv("test_residential_single_family_week3_no_outliers.csv")

print("Train shape:", train_no_outliers.shape)
print("Test shape:", test_no_outliers.shape)

Train shape: (128568, 57)
Test shape: (11914, 57)


In [7]:
numeric_features = [
    "LivingArea",
    "BedroomsTotal",
    "BathroomsTotalInteger",
    "LotSizeSquareFeet",
    "LotSizeAcres",
    "LotSizeArea",
    "YearBuilt",
    "GarageSpaces",
    "ParkingTotal",
    "Latitude",
    "Longitude",
    "Stories",
    "AssociationFee",
    "MainLevelBedrooms"
]

categorical_features = [
    "CountyOrParish",
    "City",
    "PostalCode",
    "HighSchoolDistrict",
    "Flooring",
    "AttachedGarageYN",
    "Levels",
    "PoolPrivateYN",
    "NewConstructionYN",
    "ViewYN",
    "FireplaceYN"
]

In [9]:
X_train_raw_no_outliers = train_no_outliers[numeric_features + categorical_features].copy()
X_test_raw_no_outliers = test_no_outliers[numeric_features + categorical_features].copy()

y_train_no_outliers = train_no_outliers["ClosePrice"].copy()
y_test_no_outliers = test_no_outliers["ClosePrice"].copy()

print("X_train_raw_no_outliers shape:", X_train_raw_no_outliers.shape)
print("X_test_raw_no_outliers shape:", X_test_raw_no_outliers.shape)
print("y_train_no_outliers shape:", y_train_no_outliers.shape)
print("y_test_no_outliers shape:", y_test_no_outliers.shape)

X_train_raw_no_outliers shape: (128568, 25)
X_test_raw_no_outliers shape: (11914, 25)
y_train_no_outliers shape: (128568,)
y_test_no_outliers shape: (11914,)


In [13]:
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer

# Make categorical columns strings
for col in categorical_features:
    X_train_raw_no_outliers[col] = X_train_raw_no_outliers[col].astype(str)
    X_test_raw_no_outliers[col] = X_test_raw_no_outliers[col].astype(str)

try:
    encoder_no_outliers = OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=True
    )
except TypeError:
    encoder_no_outliers = OneHotEncoder(
        handle_unknown="ignore",
        sparse=True
    )

preprocessor_no_outliers = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(with_mean=False), numeric_features),
        ("cat", encoder_no_outliers, categorical_features)
    ],
    remainder="drop"
)

X_train_processed_no_outliers = preprocessor_no_outliers.fit_transform(X_train_raw_no_outliers)
X_test_processed_no_outliers = preprocessor_no_outliers.transform(X_test_raw_no_outliers)

print("Processed train shape:", X_train_processed_no_outliers.shape)
print("Processed test shape:", X_test_processed_no_outliers.shape)
print("Data type:", type(X_train_processed_no_outliers))

Processed train shape: (128568, 3871)
Processed test shape: (11914, 3871)
Data type: <class 'scipy.sparse._csr.csr_matrix'>


In [14]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

linear_no_outliers = LinearRegression()

linear_no_outliers.fit(X_train_processed_no_outliers, y_train_no_outliers)

linear_pred_no_outliers = linear_no_outliers.predict(X_test_processed_no_outliers)

print("Linear Regression No-Outlier model trained.")
print("Number of predictions:", len(linear_pred_no_outliers))
print("First 5 predictions:", linear_pred_no_outliers[:5])

Linear Regression No-Outlier model trained.
Number of predictions: 11914
First 5 predictions: [ 805077.65112655  572317.05918841  257051.58275375 2044558.84623138
  632909.04244095]


#### **Evaluate the model**

In [15]:
linear_no_outlier_r2 = r2_score(y_test_no_outliers, linear_pred_no_outliers)
linear_no_outlier_mae = mean_absolute_error(y_test_no_outliers, linear_pred_no_outliers)
linear_no_outlier_rmse = np.sqrt(mean_squared_error(y_test_no_outliers, linear_pred_no_outliers))

print("Linear Regression No-Outlier Results")
print("R²:", linear_no_outlier_r2)
print("MAE:", linear_no_outlier_mae)
print("RMSE:", linear_no_outlier_rmse)

Linear Regression No-Outlier Results
R²: 0.8415249235130129
MAE: 220162.60581909068
RMSE: 360590.0426259625


In [17]:
baseline_results = pd.DataFrame({
    "Model": ["Linear Regression"],
    "Training Window": ["12 months"],
    "Test Month": ["2026-05"],
    "R2": [linear_no_outlier_r2],
    "MAE": [linear_no_outlier_mae],
    "RMSE": [linear_no_outlier_rmse]
})

baseline_results

,Model,Training Window,Test Month,R2,MAE,RMSE
0,Linear Regression,12 months,2026-05,0.841525,220162.605819,360590.042626


After removing the top 1% most expensive properties using a cutoff learned from the training set only, the Linear Regression model improved substantially. The R² increased from 0.287 to 0.842. This suggests that extreme high-price properties were a major source of prediction error in the original model.

The no-outlier result should be interpreted as a separate experiment rather than a replacement for the main baseline. The original model evaluates performance on all single-family homes, while the no-outlier experiment evaluates performance only on homes below approximately $6.42M. This shows that the model performs much better on the more typical price range but struggles with luxury or unusual properties.